# Notebook 00 — Data Loading

**Purpose:** Pull and verify data from 4 sources into the raw data folder.

## Data Sources

| Source | Series Count | Purpose |
|---|---|---|
| **yfinance** | 4 | Price data (EUR/USD, DXY, Gold, Oil) |
| **FRED** | 13 | US/EU macroeconomic indicators |
| **ECB** | 4 | EU rates + EUR Effective Exchange Rate |
| **COT** | 1 | EUR speculator positioning |

**Total:** 22 series from 4 independent sources

## Economic Theory Behind Each Source

1. **Interest Rate Parity (IRP)**: US-EU rate differential drives capital flow → FX
2. **Purchasing Power Parity (PPP)**: Inflation differential is a long-term driver
3. **Risk Sentiment**: High VIX → flight-to-USD → EUR/USD falls
4. **Behavioral**: COT positioning of smart money (large speculators)
5. **Cross-validation**: 3 EUR/USD sources (yfinance, FRED, ECB) to detect data bugs

---

**Author:** Dong Cong Gia Khang  
**Date:** April 2026

## 1. Setup

Import libraries and configure project root path to enable imports from `src/`.

In [10]:
"""
Setup project root to enable imports from src/
"""
import sys
from pathlib import Path

# Go up 1 level from notebooks/ to project root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")

# Verify .env exists
env_path = PROJECT_ROOT / ".env"
print(f".env exists: {env_path.exists()}")

Project root: d:\Final Project 2
Python version: 3.14.4
.env exists: True


## 2. yfinance — Price Data

Pull 4 assets from Yahoo Finance:
- **EUR/USD** (`EURUSD=X`) — target asset to forecast
- **DXY** (`DX-Y.NYB`) — US Dollar Index, inversely correlated with EUR/USD by construction
- **Gold** (`GC=F`) — safe haven, correlates with risk sentiment
- **Oil** (`CL=F`) — commodity benchmark

> **Note:** Files already pulled and saved in `data/raw/yfinance/`. The cell below verifies existing files.

In [11]:
"""
Check yfinance data already pulled.
"""
yfinance_dir = PROJECT_ROOT / "data" / "raw" / "yfinance"
yfinance_files = list(yfinance_dir.glob("*.csv"))

print(f"yfinance files: {len(yfinance_files)}")
for f in yfinance_files:
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name:15s}  ({size_kb:.1f} KB)")

yfinance files: 4
  - dxy.csv          (118.8 KB)
  - eurusd.csv       (127.9 KB)
  - gold.csv         (112.8 KB)
  - oil.csv          (119.2 KB)


## 3. FRED — Macro Indicators

13 macroeconomic series from Federal Reserve Economic Data:

| Category | Series |
|---|---|
| US Rates (3) | DFF, DGS2, DGS10 |
| EU Rates (2) | ECBDFR, IRLTLT01EZM156N |
| Inflation (3) | CPIAUCSL, CP0000EZ19M086NEST, T10YIE |
| Macro Health (2) | UNRATE, PAYEMS |
| Risk Sentiment (1) | VIXCLS |
| USD Strength (1) | DTWEXBGS |
| Cross-check (1) | DEXUSEU |

**Theory:** Interest Rate Parity (IRP) + Purchasing Power Parity (PPP)

In [12]:
"""
Check and pull FRED data if not already present.
"""
from src.data.load_fred import download_all as fred_download

fred_dir = PROJECT_ROOT / "data" / "raw" / "fred"
existing_files = list(fred_dir.glob("*.csv"))

if len(existing_files) >= 13:
    print(f"[OK] FRED already has {len(existing_files)} files. Skip download.")
    for f in sorted(existing_files):
        size_kb = f.stat().st_size / 1024
        print(f"  - {f.name:30s}  ({size_kb:.1f} KB)")
else:
    print(f"FRED has only {len(existing_files)} files. Pulling...")
    fred_download()

ModuleNotFoundError: No module named 'fredapi'

## 4. ECB — European Central Bank

4 series from ECB Data Portal (SDMX API, no API key required):

- **EUR/USD official** — cross-check with yfinance and FRED
- **EUR Effective Exchange Rate (EER-40)** — EUR strength vs basket of 40 trading partners
- **Euribor 3M** — EU benchmark rate
- **ESTER overnight** — replaced EONIA from October 2019

**Theory:** Cross-validation + EU-specific data not available in FRED

In [ ]:
"""
Check and pull ECB data if not already present.
"""
from src.data.load_ecb import download_all as ecb_download

ecb_dir = PROJECT_ROOT / "data" / "raw" / "ecb"
existing_files = list(ecb_dir.glob("*.csv"))

if len(existing_files) >= 4:
    print(f"[OK] ECB already has {len(existing_files)} files. Skip download.")
    for f in sorted(existing_files):
        size_kb = f.stat().st_size / 1024
        print(f"  - {f.name:30s}  ({size_kb:.1f} KB)")
else:
    print(f"ECB has only {len(existing_files)} files. Pulling...")
    ecb_download()

[OK] ECB already has 4 files. Skip download.
  - ester_overnight.csv             (30.2 KB)
  - eur_effective_rate.csv          (3.5 KB)
  - euribor_3m.csv                  (3.6 KB)
  - eurusd_official.csv             (77.3 KB)


## 5. COT — CFTC Commitment of Traders

Weekly data on **large speculator (non-commercial)** positioning in EUR futures.

**Economic Logic:**
- `eur_net_position = long - short` (raw positioning)
- `eur_net_position_pct` = normalized to [-1, +1] (removes dependency on total volume)
- **Extreme positioning = contrarian signal**:
  - Extreme Net Long → bullish exhausted → potential reversal down
  - Extreme Net Short → bearish exhausted → potential reversal up

**References:**
- Sanders et al. (2004) — "COT and futures trends prediction"
- CFTC official: https://www.cftc.gov/MarketReports/CommitmentsofTraders

In [ ]:
"""
Check and pull COT data if not already present.

Note: COT takes ~1-2 minutes on first pull (downloads zip files from CFTC).
"""
from src.data.load_cot import download_cot_data

cot_dir = PROJECT_ROOT / "data" / "raw" / "cot"
cot_file = cot_dir / "eur_cot.csv"

if cot_file.exists():
    print(f"[OK] COT already exists. Skip download.")
    size_kb = cot_file.stat().st_size / 1024
    print(f"  - {cot_file.name}  ({size_kb:.1f} KB)")
else:
    print(f"COT not found. Pulling (will take 1-2 minutes)...")
    download_cot_data()

[OK] COT already exists. Skip download.
  - eur_cot.csv  (77.3 KB)


## 6. Summary — Total Data Pulled

Aggregate all raw data, check coverage and file sizes.

In [ ]:
"""
Summary of all raw data pulled.
"""
import pandas as pd

raw_dir = PROJECT_ROOT / "data" / "raw"

summary = []
for source_dir in sorted(raw_dir.iterdir()):
    if not source_dir.is_dir():
        continue
    files = list(source_dir.glob("*.csv"))
    total_size_kb = sum(f.stat().st_size for f in files) / 1024
    
    summary.append({
        "Source": source_dir.name,
        "Num Files": len(files),
        "Total Size (KB)": round(total_size_kb, 1),
    })

df_summary = pd.DataFrame(summary)
print("RAW DATA SUMMARY")
print("=" * 60)
print(df_summary.to_string(index=False))
print("=" * 60)
print(f"TOTAL: {df_summary['Num Files'].sum()} files, "
      f"{df_summary['Total Size (KB)'].sum():.1f} KB")

RAW DATA SUMMARY
  Source  Num Files  Total Size (KB)
     cot          1             77.3
     ecb          4            114.6
    fred         13            660.4
     imf          0              0.0
yfinance          4           1374.1
TOTAL: 22 files, 2226.4 KB


## Conclusion

### Summary

 **22 series loaded** from 4 independent sources  
 All files verified present in `data/raw/`

### Data Sources Recap

| Source | Files | Role |
|---|---|---|
| yfinance | 4 | Price data (target + context) |
| FRED | 13 | US/EU macro indicators |
| ECB | 4 | European-specific data |
| COT | 1 | Speculator positioning |

**Total: 22 series, ~16 years (2010-2026)**

### Next Steps

This notebook only **loads and verifies** the data. Detailed inspection of each source is done in dedicated notebooks:

1. **`00a_inspect_yfinance.ipynb`** — 4 price series (EUR/USD, DXY, Gold, Oil)
2. **`00b_inspect_fred.ipynb`** — 13 macroeconomic series
3. **`00c_inspect_ecb.ipynb`** — 4 European series
4. **`00d_inspect_cot.ipynb`** — COT positioning data

Each series is examined individually with statistics, time-series plots, and distribution analysis. Charts are saved to `figures/data_inspection/`.

After understanding each series, the master dataset is built and quality-checked in `01_data_quality.ipynb`.